## 1. Data Understanding

In [1]:
import pandas as pd
import numpy as np
df_2009_2010 = pd.read_excel('../data/raw/online_retail_II.xlsx', sheet_name='Year 2009-2010')
df_2010_2011 = pd.read_excel('../data/raw/online_retail_II.xlsx', sheet_name='Year 2010-2011')
df = pd.concat([df_2009_2010, df_2010_2011], ignore_index=True)

df.to_csv('../data/raw/online_retail_combined.csv', index=False)

FileNotFoundError: [Errno 2] No such file or directory: '../data/raw/online_retail_II.xlsx'

In [ ]:
print(df.shape)
df.head()

In [ ]:
df. info

In [ ]:
df. duplicated().sum()

In [ ]:
df .isnull(). sum()

In [ ]:
missing = df .isnull(). sum()
missing_pct = (missing / len(df)) *100
pd.DataFrame({'missing_count': missing, 'missing_pct': missing_pct.round(2)})

In [ ]:
print(df.columns.tolist())

In [ ]:
df[['Quantity', 'Price']].describe()

In [ ]:
df['InvoiceDate'].min() , df['InvoiceDate'].max()

In [ ]:
for col in ['Customer ID', 'StockCode', 'Invoice', 'Country']:
    print(col, '->', df[col].nunique())

In [ ]:
cancelled = df[df['Invoice'].astype(str).str.startswith('C')]
print(f"{len(cancelled)} cancelled transactions, {len(cancelled)/len(df)*100:.2f}% of all rows")

In [ ]:
(df['Price'] <= 0).sum()

In [ ]:
df[['Quantity', 'Price']].describe()

In [ ]:
df[['Description', 'Country', 'Invoice', 'StockCode']].describe()

In [ ]:
df['Country'].value_counts()

In [ ]:
# Top 20 most frequent products
df['Description'].value_counts().head(20)

In [ ]:
df['Country'].value_counts(normalize=True) * 100

In [ ]:
df['StockCode'].value_counts().head(20)

In [ ]:
df['Invoice'].value_counts().describe()

In [ ]:
# Negative Quantity (returns/cancellations)
neg_qty = (df['Quantity'] < 0).sum()
zero_qty = (df['Quantity'] == 0).sum()
print(f"Negative Quantity: {neg_qty} rows ({neg_qty/len(df)*100:.2f}%)")
print(f"Zero Quantity: {zero_qty} rows ({zero_qty/len(df)*100:.2f}%)")

In [ ]:
# Negative and zero Price
neg_price = (df['Price'] < 0).sum()
zero_price = (df['Price'] == 0).sum()
print(f"Negative Price: {neg_price} rows ({neg_price/len(df)*100:.2f}%)")
print(f"Zero Price: {zero_price} rows ({zero_price/len(df)*100:.2f}%)")

In [ ]:
# Look at the actual rows with the most extreme Quantity values
df.nlargest(10, 'Quantity')[['Invoice', 'StockCode', 'Description', 'Quantity', 'Price', 'Customer ID']]
df.nsmallest(10, 'Quantity')[['Invoice', 'StockCode', 'Description', 'Quantity', 'Price', 'Customer ID']]

# Same for Price
df.nlargest(10, 'Price')[['Invoice', 'StockCode', 'Description', 'Quantity', 'Price', 'Customer ID']]

In [ ]:
# Find all non-numeric-looking StockCodes — real products are typically 5-digit codes, sometimes with a letter suffix
for code in ['M', 'BANK CHARGES', 'AMAZONFEE', 'POST', 'DOT', 'D', 'CRUK', 'C2', 'CRUK', 'B', 'ADJUST', 'ADJUST2']:
    count = (df['StockCode'] == code).sum()
    if count > 0:
        print(f"{code}: {count} rows")       

In [ ]:
df.nlargest(5, 'Quantity')[['Invoice', 'StockCode', 'Description', 'Quantity', 'Price', 'Customer ID']]

In [ ]:
df.nsmallest(5, 'Quantity')[['Invoice', 'StockCode', 'Description', 'Quantity', 'Price', 'Customer ID']]

The combined dataset contains 1,067,371 rows across 8 columns, spanning Dec 2009–Dec 2011. The business is heavily UK-concentrated: 91.9% of transactions originate from the United Kingdom, with the remaining 8.1% spread across 42 other countries, plus 756 rows (0.07%) labeled "Unspecified." Customer ID is missing in 22.7% of rows — these transactions still represent real revenue but can't be attributed to a specific customer. 2.15% of rows have negative Quantity (22,950 rows), corresponding to cancellations/returns; Quantity also contains extreme outliers (min -80,995, max 80,995), which will require investigation beyond a simple sign check. Price has 5 negative values (negligible) and 6,202 zero values (0.58%), likely representing free items or adjustments; Price also has a right-skewed outlier at its max (38,970) relative to its 75th percentile (4.15). The "POSTAGE" line item appears 2,115 times as a top "product," representing shipping charges rather than real merchandise and requiring separate handling. Median items per invoice is 9, but with a long tail up to 1,350 items on a single invoice, consistent with a mix of individual retail purchases and bulk wholesale orders.

### 2. Data Cleaning

In [ ]:
#Convert data types
df ['InvoiceDate'] = pd.to_datetime(df['InvoiceDate'])
df ['StockCode'] = df['StockCode'].astype(str)
df ['Invoice'] = df['Invoice'].astype(str)

In [ ]:
#Handle missing rows
missing_customer = df['Customer ID'].isnull().sum()
print(f"{missing_customer} rows ({missing_customer/len(df)*100: .2f}%) missing Customer ID")
df_with_customer = df.dropna(subset=['Customer ID']).copy()
df_with_customer['Customer ID'] = df_with_customer['Customer ID'].astype(int)                                                         

In [ ]:
#Seperate cancellations 
df['is_cancelled'] = df['Invoice'].str.startswith('C')
cancelled_count = df['is_cancelled'].sum()
df_sales = df[~df['is_cancelled']].copy()
df_cancellations = df[df['is_cancelled']].copy()

In [ ]:
#Remove non product administrative codes 
non_product_codes = ['POST', 'DOT', 'M', 'C2', 'D', 'BANK CHARGES', 'AMAZONFEE', 'CRUK', 'B', 'ADJUST', 'ADJUST2']

removed_admin = df_sales['StockCode'].isin(non_product_codes).sum()
print(f"{removed_admin} rows removed as non-product administrative entries")

df_sales = df_sales[~df_sales['StockCode'].isin(non_product_codes)].copy()

In [ ]:
#Remove zero/negative Price
invalid_price = (df_sales['Price'] <= 0).sum()
print(f"{invalid_price} rows with zero or negative Price removed")

df_sales = df_sales[df_sales['Price'] > 0].copy()

In [ ]:
#Handle missing Description
missing_desc = df_sales['Description'].isnull().sum()
print(f"{missing_desc} rows with missing Description filled as 'Unknow Product'")

df_sales['Description'] = df_sales['Description'].fillna('Unknown Product')

In [ ]:
#remove duplicate rows
dupes = df_sales.duplicated().sum()
print(f"{dupes} exact duplicate rows removed")

df_sales = df_sales.drop_duplicates().copy()

In [ ]:
#confirm no negative Quantity slipped through
remaining_negative_qty = (df_sales ['Quantity'] < 0).sum()
print(f"Remaining negative Quantity rows in df_sales: {remaining_negative_qty}")

In [ ]:
#Create Revenue column
df_sales['Revenue'] = df_sales['Quantity'] * df_sales['Price']

## 3. Feature Engineering: Product Category

In [ ]:
df_sales['Description'].value_counts().head(60)

In [ ]:
def categorize_product(description):
    desc = str(description).upper()

    if any(word in desc for word in ['CHRISTMAS', 'XMAS', 'ADVENT']):
        return 'Christmas & Seasonal'
    elif any(word in desc for word in ['BAG', 'SHOPPER', 'BASKET']):
      return 'Bags & Storage'
    elif any(word in desc for word in ['CAKE', 'BAKING', 'TEATIME', 'MUG', 'TEACUP', 'SAUCER', 'SNACK BOX', 'HOT WATER BOTTLE']):
      return 'Kitchen & Dining '
    elif any(word in desc for word in ['LIGHT HOLDER', 'T-LIGHT', 'CANDLE', 'ORNAMENT', 'FRAME', 'CHALKBOARD', 'NIGHT LIGHT', 'DECORATION']):
      return 'Home Decor & Lighting'
    elif any(word in desc for word in ['SIGNS']):
      return 'Novelty Signs'
    elif any(word in desc for word in ['BUNTING', 'PARTY', 'BALLOON']):
      return 'Party & Occasion'
    elif any(word in desc for word in ['TRINKET BOX', 'RECIPE BOX', 'BOX']):
        return 'Storage & Boxes'
    elif any(word in desc for word in ['CARD', 'WRAP', 'GIFT TAG', 'RIBBON', 'PAPER CHAIN'] ):
        return 'Stationery & Gift Wrap'
    elif any(word in desc for word in['FELT', 'CRAFT', 'WICKER']):
        return 'Craft & Textile'
    else:
        return 'Other'      

df_sales['Category'] = df_sales['Description'].apply(categorize_product)

In [ ]:
df_sales['Category'].value_counts()

In [ ]:
df_sales['Category'].value_counts(normalize=True) * 100

In [ ]:
df_sales[df_sales['Category'] == 'Other']['Description'].value_counts().head(40)

In [ ]:
df_sales['Category'] = df_sales['Description'].apply(categorize_product)
df_sales[df_sales['Category'] == 'Other']['Description'].value_counts().head(40)

In [ ]:
def categorize_product(description):
    desc = str(description).upper()
    
    if any(word in desc for word in ['CHRISTMAS', 'XMAS', 'ADVENT']):
        return 'Christmas & Seasonal'
    elif any(word in desc for word in ['BAG', 'SHOPPER', 'BASKET']):
        return 'Bags & Storage'
    elif any(word in desc for word in ['CAKE', 'BAKING', 'TEATIME', 'MUG', 'TEACUP', 'SAUCER',
                                         'SNACK BOX', 'HOT WATER BOTTLE', 'KITCHEN SCALES',
                                         'JAM MAKING', 'COOKIE CUTTER', 'NAPKIN', 'LID GLASS BOWL',
                                         'JELLY MOULD', 'TEA SET', 'TEA GLASS']):
        return 'Kitchen & Dining'
    elif any(word in desc for word in ['LIGHT HOLDER', 'T-LIGHT', 'CANDLE', 'ORNAMENT', 'FRAME',
                                         'CHALKBOARD', 'NIGHT LIGHT', 'DECORATION', 'CHILLI LIGHTS']):
        return 'Home Decor & Lighting'
    elif 'SIGN' in desc:
        return 'Novelty Signs'
    elif any(word in desc for word in ['BUNTING', 'PARTY', 'BALLOON']):
        return 'Party & Occasion'
    elif any(word in desc for word in ['TRINKET BOX', 'RECIPE BOX', 'TIN', 'BOX', 'CABINET', 'COAT RACK']):
        return 'Storage & Boxes'
    elif any(word in desc for word in ['CARD', 'WRAP', 'GIFT TAG', 'RIBBON', 'PAPER CHAIN']):
        return 'Stationery & Gift Wrap'
    elif any(word in desc for word in ['FELT', 'CRAFT', 'WICKER']):
        return 'Craft & Textile'
    elif any(word in desc for word in ['BUILDING BLOCK', 'ALARM CLOCK', 'HAND WARMER', 'PLASTERS',
                                         'PARASOL', 'PEG']):
        return 'Novelty & Personal Items'
    else:
        return 'Other'

df_sales['Category'] = df_sales['Description'].apply(categorize_product)
df_sales['Category'].value_counts(normalize=True) * 100

In [ ]:
df_sales[df_sales['Category'] == 'Other']['Description'].value_counts().head(40)

In [ ]:
def categorize_product(description):
    desc = str(description).upper()
    
    if any(word in desc for word in ['CHRISTMAS', 'XMAS', 'ADVENT']):
        return 'Christmas & Seasonal'
    elif any(word in desc for word in ['BAG', 'SHOPPER', 'BASKET']):
        return 'Bags & Storage'
    elif any(word in desc for word in ['CAKE', 'BAKING', 'TEATIME', 'MUG', 'TEACUP', 'SAUCER',
                                         'SNACK BOX', 'HOT WATER BOTTLE', 'KITCHEN SCALES',
                                         'JAM MAKING', 'COOKIE CUTTER', 'NAPKIN', 'LID GLASS BOWL',
                                         'JELLY MOULD', 'TEA SET', 'TEA GLASS', 'MEASURING SPOONS',
                                         'MILK JUG', 'DOILIES']):
        return 'Kitchen & Dining'
    elif any(word in desc for word in ['LIGHT HOLDER', 'T-LIGHT', 'CANDLE', 'ORNAMENT', 'FRAME',
                                         'CHALKBOARD', 'NIGHT LIGHT', 'DECORATION', 'CHILLI LIGHTS',
                                         'POPCORN HOLDER']):
        return 'Home Decor & Lighting'
    elif 'SIGN' in desc:
        return 'Novelty Signs'
    elif any(word in desc for word in ['BUNTING', 'PARTY', 'BALLOON']):
        return 'Party & Occasion'
    elif any(word in desc for word in ['TRINKET', 'RECIPE BOX', 'TIN', 'BOX', 'CABINET', 'COAT RACK']):
        return 'Storage & Boxes'
    elif any(word in desc for word in ['CARD', 'WRAP', 'GIFT TAG', 'RIBBON', 'PAPER CHAIN',
                                         'NOTEBOOK', 'TISSUE', 'PENCIL']):
        return 'Stationery & Gift Wrap'
    elif any(word in desc for word in ['FELT', 'CRAFT', 'WICKER']):
        return 'Craft & Textile'
    elif any(word in desc for word in ['BUILDING BLOCK', 'ALARM CLOCK', 'HAND WARMER', 'PLASTERS',
                                         'PARASOL', 'PEG', 'KEY FOB', 'SEWING KIT', 'FAN']):
        return 'Novelty & Personal Items'
    elif any(word in desc for word in ['DOORMAT', 'GARDEN', 'PLANT LADDER', 'KNEELING PAD']):
        return 'Garden & Outdoor'
    elif any(word in desc for word in ['HOOK', 'HANGER', 'DRAWER KNOB']):
        return 'Hardware & Fixtures'
    elif any(word in desc for word in ['SPINNING TOP', 'SKIPPING ROPE', 'SNAKES & LADDERS',
                                         'BINGO', 'CATCH CUP', 'TOY', 'GAME']):
        return 'Toys & Games'
    else:
        return 'Other'

df_sales['Category'] = df_sales['Description'].apply(categorize_product)
df_sales['Category'].value_counts(normalize=True) * 100

In [ ]:
df_sales[df_sales['Category'] == 'Other']['Description'].value_counts().head(40)

In [ ]:
other_revenue_pct = df_sales[df_sales['Category'] == 'Other']['Revenue'].sum() / df_sales['Revenue'].sum() * 100
print(f"'Other' represents {other_revenue_pct:.2f}% of total revenue")

In [ ]:
def categorize_product(description):
    desc = str(description).upper()
    
    if any(word in desc for word in ['CHRISTMAS', 'XMAS', 'ADVENT']):
        return 'Christmas & Seasonal'
    elif any(word in desc for word in ['BAG', 'SHOPPER', 'BASKET']):
        return 'Bags & Storage'
    elif any(word in desc for word in ['CAKE', 'BAKING', 'TEATIME', 'MUG', 'TEACUP', 'SAUCER',
                                         'SNACK BOX', 'HOT WATER BOTTLE', 'KITCHEN SCALES',
                                         'JAM MAKING', 'COOKIE CUTTER', 'NAPKIN', 'LID GLASS BOWL',
                                         'JELLY MOULD', 'TEA SET', 'TEA GLASS', 'MEASURING SPOONS',
                                         'MILK JUG', 'DOILIES', 'ENAMEL', 'OVEN GLOVE', 'BREAD BIN',
                                         'FRYING PAN', 'BREAKFAST PLATE', 'PAPER PLATE', 'PAPER CUP',
                                         'FOOD CONTAINER']):
        return 'Kitchen & Dining'
    elif any(word in desc for word in ['LIGHT HOLDER', 'T-LIGHT', 'CANDLE', 'ORNAMENT', 'FRAME',
                                         'CHALKBOARD', 'NIGHT LIGHT', 'DECORATION', 'CHILLI LIGHTS',
                                         'POPCORN HOLDER', 'LANTERN', 'DOORSTOP', 'CUSHION COVER',
                                         'HEART']):
        return 'Home Decor & Lighting'
    elif 'SIGN' in desc:
        return 'Novelty Signs'
    elif any(word in desc for word in ['BUNTING', 'PARTY', 'BALLOON']):
        return 'Party & Occasion'
    elif any(word in desc for word in ['TRINKET', 'RECIPE BOX', 'TIN', 'BOX', 'CABINET', 'COAT RACK',
                                         'MINI CHEST']):
        return 'Storage & Boxes'
    elif any(word in desc for word in ['CARD', 'WRAP', 'GIFT TAG', 'RIBBON', 'PAPER CHAIN',
                                         'NOTEBOOK', 'TISSUE', 'PENCIL', 'STATIONERY']):
        return 'Stationery & Gift Wrap'
    elif any(word in desc for word in ['FELT', 'CRAFT', 'WICKER']):
        return 'Craft & Textile'
    elif any(word in desc for word in ['BUILDING BLOCK', 'ALARM CLOCK', 'HAND WARMER', 'PLASTERS',
                                         'PARASOL', 'PEG', 'KEY FOB', 'SEWING KIT', 'FAN', 'APRON',
                                         'HAIR CLIP', 'GLITTER']):
        return 'Novelty & Personal Items'
    elif any(word in desc for word in ['DOORMAT', 'DOOR MAT', 'GARDEN', 'PLANT LADDER', 'KNEELING PAD']):
        return 'Garden & Outdoor'
    elif any(word in desc for word in ['HOOK', 'HANGER', 'DRAWER KNOB']):
        return 'Hardware & Fixtures'
    elif any(word in desc for word in ['SPINNING TOP', 'SKIPPING ROPE', 'SNAKES & LADDERS',
                                         'BINGO', 'CATCH CUP', 'TOY', 'GAME', 'COLOURING',
                                         'CRAYON', 'CHALK STICK', 'MODELLING CLAY', 'PAINT SET',
                                         'DOMINOES', 'SKITTLES', 'PLAYHOUSE']):
        return 'Toys & Kids Craft'
    else:
        return 'Other'

df_sales['Category'] = df_sales['Description'].apply(categorize_product)
df_sales['Category'].value_counts(normalize=True) * 100

In [ ]:
df_sales['Category'] = df_sales['Description'].apply(categorize_product)

print(df_sales['Category'].value_counts(normalize=True) * 100)

other_revenue_pct = df_sales[df_sales['Category'] == 'Other']['Revenue'].sum() / df_sales['Revenue'].sum() * 100
print(f"'Other' represents {other_revenue_pct:.2f}% of total revenue")

In [ ]:
df_sales[df_sales['Category'] == 'Other'].groupby('Description')['Revenue'].sum().sort_values(ascending=False).head(30)

In [ ]:
def categorize_product(description):
    desc = str(description).upper()
    
    if any(word in desc for word in ['CHRISTMAS', 'XMAS', 'ADVENT']):
        return 'Christmas & Seasonal'
    elif any(word in desc for word in ['BAG', 'SHOPPER', 'BASKET']):
        return 'Bags & Storage'
    elif any(word in desc for word in ['CAKE', 'BAKING', 'TEATIME', 'MUG', 'TEACUP', 'SAUCER',
                                         'SNACK BOX', 'HOT WATER BOTTLE', 'KITCHEN SCALES',
                                         'JAM MAKING', 'COOKIE CUTTER', 'NAPKIN', 'LID GLASS BOWL',
                                         'JELLY MOULD', 'TEA SET', 'TEA GLASS', 'MEASURING SPOONS',
                                         'MILK JUG', 'DOILIES', 'ENAMEL', 'OVEN GLOVE', 'BREAD BIN',
                                         'FRYING PAN', 'BREAKFAST PLATE', 'PAPER PLATE', 'PAPER CUP',
                                         'FOOD CONTAINER', 'TEAPOT', 'COFFEE SET', 'TEA TOWEL',
                                         'CUTLERY', 'BREAKFAST SET']):
        return 'Kitchen & Dining'
    elif any(word in desc for word in ['LIGHT HOLDER', 'T-LIGHT', 'CANDLE', 'ORNAMENT', 'FRAME',
                                         'CHALKBOARD', 'NIGHT LIGHT', 'DECORATION', 'CHILLI LIGHTS',
                                         'POPCORN HOLDER', 'LANTERN', 'DOORSTOP', 'CUSHION COVER',
                                         'HEART', 'MEMOBOARD', 'BIRDCAGE', 'PLANT HOLDER',
                                         'CHERRY LIGHTS', 'WALL CLOCK']):
        return 'Home Decor & Lighting'
    elif 'SIGN' in desc:
        return 'Novelty Signs'
    elif any(word in desc for word in ['BUNTING', 'PARTY', 'BALLOON']):
        return 'Party & Occasion'
    elif any(word in desc for word in ['TRINKET', 'RECIPE BOX', 'TIN', 'BOX', 'CABINET', 'COAT RACK',
                                         'MINI CHEST', 'STORAGE JAR', 'STORAGE CUBE', 'CRATE',
                                         'MINI CASES']):
        return 'Storage & Boxes'
    elif any(word in desc for word in ['CARD', 'WRAP', 'GIFT TAG', 'RIBBON', 'PAPER CHAIN',
                                         'NOTEBOOK', 'TISSUE', 'PENCIL', 'STATIONERY']):
        return 'Stationery & Gift Wrap'
    elif any(word in desc for word in ['FELT', 'CRAFT', 'WICKER']):
        return 'Craft & Textile'
    elif any(word in desc for word in ['BUILDING BLOCK', 'ALARM CLOCK', 'HAND WARMER', 'PLASTERS',
                                         'PARASOL', 'PEG', 'KEY FOB', 'SEWING KIT', 'FAN', 'APRON',
                                         'HAIR CLIP', 'GLITTER', 'UMBRELLA', 'PURSE', 'LIP GLOSS']):
        return 'Novelty & Personal Items'
    elif any(word in desc for word in ['DOORMAT', 'DOOR MAT', 'GARDEN', 'PLANT LADDER', 'KNEELING PAD',
                                         'GROW YOUR OWN']):
        return 'Garden & Outdoor'
    elif any(word in desc for word in ['HOOK', 'HANGER', 'DRAWER KNOB']):
        return 'Hardware & Fixtures'
    elif any(word in desc for word in ['SPINNING TOP', 'SKIPPING ROPE', 'SNAKES & LADDERS',
                                         'BINGO', 'CATCH CUP', 'TOY', 'GAME', 'COLOURING',
                                         'CRAYON', 'CHALK STICK', 'MODELLING CLAY', 'PAINT SET',
                                         'DOMINOES', 'SKITTLES', 'PLAYHOUSE', 'FLYING DUCKS']):
        return 'Toys & Kids Craft'
    else:
        return 'Other'

df_sales['Category'] = df_sales['Description'].apply(categorize_product)

print(df_sales['Category'].value_counts(normalize=True) * 100)
other_revenue_pct = df_sales[df_sales['Category'] == 'Other']['Revenue'].sum() / df_sales['Revenue'].sum() * 100
print(f"'Other' represents {other_revenue_pct:.2f}% of total revenue")

In [ ]:
df_sales[df_sales['Category'] == 'Other'].groupby('Description')['Revenue'].sum().sort_values(ascending=False).head(50)

In [ ]:
df_sales[df_sales['Description'] == 'Adjust bad debt']['StockCode'].unique()

In [ ]:
df_sales[df_sales['Category'] == 'Other'].groupby('Description')['Revenue'].sum().sort_values(ascending=False).head(50)

In [ ]:
def categorize_product(description):
    desc = str(description).upper()
    
    if any(word in desc for word in ['CHRISTMAS', 'XMAS', 'ADVENT']):
        return 'Christmas & Seasonal'
    elif any(word in desc for word in ['BAG', 'SHOPPER', 'BASKET']):
        return 'Bags & Storage'
    elif any(word in desc for word in ['CAKE', 'BAKING', 'TEATIME', 'MUG', 'TEACUP', 'SAUCER',
                                         'SNACK BOX', 'HOT WATER BOTTLE', 'KITCHEN SCALES',
                                         'JAM MAKING', 'COOKIE CUTTER', 'NAPKIN', 'LID GLASS BOWL',
                                         'JELLY MOULD', 'TEA SET', 'TEA GLASS', 'MEASURING SPOONS',
                                         'MILK JUG', 'DOILIES', 'ENAMEL', 'OVEN GLOVE', 'BREAD BIN',
                                         'FRYING PAN', 'BREAKFAST PLATE', 'PAPER PLATE', 'PAPER CUP',
                                         'FOOD CONTAINER', 'TEAPOT', 'COFFEE SET', 'TEA TOWEL',
                                         'CUTLERY', 'BREAKFAST SET', 'SUGAR BOWL', 'TEA PLATE',
                                         'BUTTER DISH', 'BEURRE DISH', 'EGG CUP', 'EGG HOLDER',
                                         'ROLLING PIN', 'TEA CADDY', 'TABLE CLOTH', 'SUGAR JAM BOWL',
                                         'BOWL', 'GLASS JAR', 'CONTAINER SET', 'WASHING UP',
                                         'CHOPPING BOARD', 'PLACEMAT', 'PLATE', 'COOKING SET',
                                         'TEA,COFFEE,SUGAR', 'TUMBLERS', 'MILK PAN', 'SCISSOR']):
        return 'Kitchen & Dining'
    elif any(word in desc for word in ['LIGHT HOLDER', 'T-LIGHT', 'CANDLE', 'ORNAMENT', 'FRAME',
                                         'CHALKBOARD', 'BLACK BOARD', 'NIGHT LIGHT', 'NIGHTLIGHT', 'DECORATION',
                                         'POPCORN HOLDER', 'LANTERN', 'DOORSTOP',
                                         'CUSHION', 'HEART', 'MEMOBOARD', 'BIRDCAGE',
                                         'PLANT HOLDER', 'WALL CLOCK',
                                         'MIRROR', 'DISCO BALL',
                                         'TORCH', 'MOBILE', 'WALL ART', 'PHOTO CUBE', 'SLATE TILE',
                                         'LIGHTS', 'PAINTED METAL', 'SLEIGH BELLS', 'LAMP',
                                         'PAINTED WOOD', 'HONEYPOT', 'DRAWING SLATE', 'TV DINNER TRAY']):
        return 'Home Decor & Lighting'
    elif 'SIGN' in desc:
        return 'Novelty Signs'
    elif any(word in desc for word in ['BUNTING', 'PARTY', 'BALLOON', 'GARLAND', 'SOMBRERO', 'LEIS']):
        return 'Party & Occasion'
    elif any(word in desc for word in ['TRINKET', 'RECIPE BOX', 'TIN', 'BOX', 'CABINET', 'COAT RACK',
                                         'MINI CHEST', 'STORAGE JAR', 'STORAGE CUBE', 'CRATE',
                                         'MINI CASES', 'ROUND CONTAINER', 'MAGAZINE RACK',
                                         'SIDEBOARD', 'ORGANISER', 'DRAWER', 'WALL TIDY',
                                         'WALL MOUNTED', 'JEWELLERY DRAWER']):
        return 'Storage & Boxes'
    elif any(word in desc for word in ['CARD', 'WRAP', 'GIFT TAG', 'RIBBON', 'PAPER CHAIN',
                                         'NOTEBOOK', 'TISSUE', 'PENCIL', 'STATIONERY', 'PEN',
                                         'RULER', 'EXERCISE BOOK']):
        return 'Stationery & Gift Wrap'
    elif any(word in desc for word in ['FELT', 'CRAFT', 'WICKER', 'PATCHES', 'PAINT YOUR OWN']):
        return 'Craft & Textile'
    elif any(word in desc for word in ['BUILDING BLOCK', 'ALARM CLOCK', 'HAND WARMER', 'PLASTERS',
                                         'PARASOL', 'PEG', 'KEY FOB', 'SEWING KIT', 'FAN', 'APRON',
                                         'HAIR CLIP', 'GLITTER', 'UMBRELLA', 'PURSE', 'LIP GLOSS',
                                         'PIGGY BANK', 'BABY GIFT', 'PONCHO', 'SLIPPER', 'HAMMOCK',
                                         'BLOCK LETTERS', 'WOOD LETTERS', 'FRIDGE MAGNET', 'MAGNET',
                                         'CALCULATOR', 'MATCHES', 'FLANNEL']):
        return 'Novelty & Personal Items'
    elif any(word in desc for word in ['DOORMAT', 'DOOR MAT', 'GARDEN', 'PLANT LADDER', 'KNEELING PAD',
                                         'GROW YOUR OWN', 'WATERING CAN', 'WHEELBARROW', 'WINDMILL',
                                         'ACAPULCO MAT']):
        return 'Garden & Outdoor'
    elif any(word in desc for word in ['HOOK', 'HANGER', 'DRAWER KNOB']):
        return 'Hardware & Fixtures'
    elif any(word in desc for word in ['SPINNING TOP', 'SKIPPING ROPE', 'SNAKES & LADDERS',
                                         'BINGO', 'CATCH CUP', 'TOY', 'GAME', 'COLOURING',
                                         'CRAYON', 'CHALK STICK', 'MODELLING CLAY', 'PAINT SET',
                                         'DOMINOES', 'SKITTLES', 'PLAYHOUSE', 'FLYING DUCKS', 'LUDO',
                                         'NAUGHTS & CROSSES', 'INFLATABLE', 'GLOBE', 'TEDDY BEAR',
                                         'BEACH SPADE']):
        return 'Toys & Kids Craft'
    else:
        return 'Other'

df_sales['Category'] = df_sales['Description'].apply(categorize_product)

print(df_sales['Category'].value_counts(normalize=True) * 100)
other_revenue_pct = df_sales[df_sales['Category'] == 'Other']['Revenue'].sum() / df_sales['Revenue'].sum() * 100
print(f"'Other' represents {other_revenue_pct:.2f}% of total revenue")

In [ ]:
df_sales.to_csv('../data/processed/online_retail_cleaned.csv', index=False)
df_cancellations.to_csv('../data/processed/cancellations.csv', index=False)
df_with_customer.to_csv('../data/processed/online_retail_with_customer.csv', index=False)

print("All cleaned datasets saved to data/processed/")

## Data Cleaning Summary
The combined raw dataset (1,067,371 rows) was cleaned by: converting data types (InvoiceDate to datetime); separating 22.7% of rows with missing Customer ID into a distinct handling path (retained for revenue analysis, excluded from customer-level analysis); splitting out 1.83% cancelled transactions (Invoice starting with 'C') into a dedicated df_cancellations dataset for returns/leakage analysis; removing non-product administrative entries (postage, bank charges, Amazon fees, manual/bad-debt adjustments — 10 distinct codes identified through iterative investigation); removing 6,207 rows with zero or negative Price; filling missing Description values; and removing exact duplicate rows. A Revenue column (Quantity × Price) was computed for all downstream analysis.
## Feature Engineering: Product Category
Since the raw dataset has no product category field, a Category feature was engineered via keyword matching against product descriptions, refined over multiple iterative rounds using both frequency-based and revenue-based sampling of uncategorized items (the latter specifically to catch high-value products that low-frequency sampling could miss). The final scheme defines 14 categories. The remaining "Other" category represents 9.55% of transactions and 5.86% of revenue — a small, low-value long tail of miscellaneous gift items, confirmed via revenue-ranked verification to contain no significant uncategorized products.